# 02 — Normalization Comparison

Side-by-side visualization of raw vs. Modified Reinhard normalized patches.
Verify that normalization reduces scanner variation without destroying
tissue structure or stain identity.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from skimage import color

from config import RAW_PATCHES_DIR, NORMALIZED_PATCHES_DIR, CLASS_NAMES
from normalization import detect_tissue, compute_lab_stats, normalize_image

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

ModuleNotFoundError: No module named 'normalization'

## Before vs After: Sample Patches

For each tissue class, show raw and normalized versions side by side.

In [ ]:
def show_before_after(n_per_class=4):
    """Show raw vs normalized patches for each class."""
    classes_to_show = ["background_h", "background_e", "vessel_h", "vessel_e"]

    fig, axes = plt.subplots(len(classes_to_show), n_per_class * 2,
                             figsize=(3 * n_per_class * 2, 3 * len(classes_to_show)))

    for row, cls in enumerate(classes_to_show):
        raw_dir = Path(RAW_PATCHES_DIR) / cls
        norm_dir = Path(NORMALIZED_PATCHES_DIR) / cls
        raw_images = sorted(raw_dir.glob("*.png"))[:n_per_class]

        for col, raw_path in enumerate(raw_images):
            # Raw
            raw_img = Image.open(raw_path)
            axes[row, col * 2].imshow(raw_img)
            axes[row, col * 2].set_title("Raw" if row == 0 and col == 0 else "")
            axes[row, col * 2].axis("off")
            if col == 0:
                axes[row, 0].set_ylabel(cls, fontsize=11, fontweight="bold")

            # Normalized
            norm_path = norm_dir / raw_path.name
            if norm_path.exists():
                norm_img = Image.open(norm_path)
                axes[row, col * 2 + 1].imshow(norm_img)
            axes[row, col * 2 + 1].set_title("Normalized" if row == 0 and col == 0 else "")
            axes[row, col * 2 + 1].axis("off")

    plt.suptitle("Raw vs Modified Reinhard Normalization", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

show_before_after()

## LAB Distribution Shift

Compare how the L, a, b channel distributions change after normalization.
The goal: L* distributions should tighten across slides (less scanner
variation), while a*/b* should partially converge without collapsing
stain identity.

In [ ]:
def compare_lab_distributions():
    """Compare LAB stats before and after normalization."""
    classes = ["background_h", "background_e", "vessel_h", "vessel_e"]

    fig, axes = plt.subplots(3, len(classes), figsize=(4 * len(classes), 10))
    channels = ["L", "a", "b"]

    for col, cls in enumerate(classes):
        raw_dir = Path(RAW_PATCHES_DIR) / cls
        norm_dir = Path(NORMALIZED_PATCHES_DIR) / cls

        raw_vals = {ch: [] for ch in channels}
        norm_vals = {ch: [] for ch in channels}

        for img_path in sorted(raw_dir.glob("*.png"))[:50]:
            raw = np.array(Image.open(img_path).convert("RGB"))
            raw_lab = color.rgb2lab(raw.astype(np.float32) / 255.0)
            raw_vals["L"].append(np.mean(raw_lab[:, :, 0]))
            raw_vals["a"].append(np.mean(raw_lab[:, :, 1]))
            raw_vals["b"].append(np.mean(raw_lab[:, :, 2]))

            norm_path = norm_dir / img_path.name
            if norm_path.exists():
                norm = np.array(Image.open(norm_path).convert("RGB"))
                norm_lab = color.rgb2lab(norm.astype(np.float32) / 255.0)
                norm_vals["L"].append(np.mean(norm_lab[:, :, 0]))
                norm_vals["a"].append(np.mean(norm_lab[:, :, 1]))
                norm_vals["b"].append(np.mean(norm_lab[:, :, 2]))

        for row, ch in enumerate(channels):
            ax = axes[row, col]
            if raw_vals[ch]:
                ax.hist(raw_vals[ch], bins=15, alpha=0.5, label="Raw", color="gray")
            if norm_vals[ch]:
                ax.hist(norm_vals[ch], bins=15, alpha=0.5, label="Normalized", color="steelblue")
            if row == 0:
                ax.set_title(cls, fontsize=11, fontweight="bold")
            if col == 0:
                ax.set_ylabel(f"{ch}* channel")
            ax.legend(fontsize=8)

    plt.suptitle("LAB Distribution: Raw vs Normalized", fontsize=14)
    plt.tight_layout()
    plt.show()

compare_lab_distributions()

## Tissue Detection Visualization

Verify that the LAB-based tissue detector correctly identifies tissue
vs. slide background, especially on tricky hematoxylin-dominant patches.

In [ ]:
def show_tissue_detection(n_examples=6):
    """Visualize tissue masks on sample patches."""
    fig, axes = plt.subplots(n_examples, 3, figsize=(10, 3 * n_examples))
    axes[0, 0].set_title("Original")
    axes[0, 1].set_title("Tissue Mask")
    axes[0, 2].set_title("Masked")

    # Grab patches from different classes
    all_patches = []
    for cls in ["background_h", "vessel_e", "background_e", "vessel_h"]:
        cls_dir = Path(RAW_PATCHES_DIR) / cls
        patches = sorted(cls_dir.glob("*.png"))[:2]
        all_patches.extend([(p, cls) for p in patches])

    for i, (img_path, cls) in enumerate(all_patches[:n_examples]):
        img = np.array(Image.open(img_path).convert("RGB"))
        tissue_mask, tissue_frac = detect_tissue(img)

        axes[i, 0].imshow(img)
        axes[i, 0].set_ylabel(f"{cls}\n{tissue_frac:.1%} tissue", fontsize=9)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(tissue_mask, cmap="gray")
        axes[i, 1].axis("off")

        masked = img.copy()
        masked[~tissue_mask] = 255
        axes[i, 2].imshow(masked)
        axes[i, 2].axis("off")

    plt.suptitle("LAB-Based Tissue Detection", fontsize=14)
    plt.tight_layout()
    plt.show()

show_tissue_detection()

## Normalization on a Single Patch (Step by Step)

Walk through what Modified Reinhard does to one image.

In [ ]:
# Pick a single hematoxylin background patch
sample_path = sorted((Path(RAW_PATCHES_DIR) / "background_h").glob("*.png"))[0]
raw = np.array(Image.open(sample_path).convert("RGB"))

# Step 1: detect tissue
tissue_mask, tissue_frac = detect_tissue(raw)
print(f"Tissue fraction: {tissue_frac:.1%}")

# Step 2: compute source stats
source_stats = compute_lab_stats(raw, tissue_mask)
print(f"Source stats: L={source_stats['L_mean']:.1f}±{source_stats['L_std']:.1f}, "
      f"a={source_stats['a_mean']:.1f}±{source_stats['a_std']:.1f}, "
      f"b={source_stats['b_mean']:.1f}±{source_stats['b_std']:.1f}")

# Step 3: normalize
normalized = normalize_image(raw)

# Step 4: check output stats
norm_stats = compute_lab_stats(normalized, tissue_mask)
print(f"Output stats:  L={norm_stats['L_mean']:.1f}±{norm_stats['L_std']:.1f}, "
      f"a={norm_stats['a_mean']:.1f}±{norm_stats['a_std']:.1f}, "
      f"b={norm_stats['b_mean']:.1f}±{norm_stats['b_std']:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw)
axes[0].set_title(f"Raw — L*={source_stats['L_mean']:.0f}, b*={source_stats['b_mean']:.0f}")
axes[0].axis("off")
axes[1].imshow(normalized)
axes[1].set_title(f"Normalized — L*={norm_stats['L_mean']:.0f}, b*={norm_stats['b_mean']:.0f}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Key Takeaways

- [ ] Normalized patches look visually consistent across slides
- [ ] Tissue structure preserved (vessels still clearly visible)
- [ ] Hematoxylin patches stay blue-purple (not shifted to pink)
- [ ] Eosin patches stay pink (not shifted to blue)
- [ ] Tissue detector correctly excludes white background